In [1]:
import pandas as pd
import numpy as np
import os
import urllib
from dotenv import load_dotenv
from sqlalchemy import create_engine


In [ ]:
# Load raw CSV

df = pd.read_csv("../data/supply_chain_dataset1.csv", parse_dates=['Date'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91250 entries, 0 to 91249
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Date                     91250 non-null  datetime64[ns]
 1   SKU_ID                   91250 non-null  object        
 2   Warehouse_ID             91250 non-null  object        
 3   Supplier_ID              91250 non-null  object        
 4   Region                   91250 non-null  object        
 5   Units_Sold               91250 non-null  int64         
 6   Inventory_Level          91250 non-null  int64         
 7   Supplier_Lead_Time_Days  91250 non-null  int64         
 8   Reorder_Point            91250 non-null  int64         
 9   Order_Quantity           91250 non-null  int64         
 10  Unit_Cost                91250 non-null  float64       
 11  Unit_Price               91250 non-null  float64       
 12  Promotion_Flag           91250 n

In [ ]:
# Rename to consistent snake_case

df = df.rename(columns={
    'Date': 'sale_date',
    'SKU_ID': 'sku_id',
    'Warehouse_ID': 'warehouse_id',
    'Supplier_ID': 'supplier_id',
    'Region': 'region',
    'Units_Sold': 'units_sold',
    'Inventory_Level': 'inventory_level',
    'Supplier_Lead_Time_Days': 'supplier_lead_time_days',
    'Reorder_Point': 'reorder_point',
    'Order_Quantity': 'order_quantity',
    'Unit_Cost': 'unit_cost',
    'Unit_Price': 'unit_price',
    'Promotion_Flag': 'is_promotion',
    'Stockout_Flag': 'stockout_flag_raw',   
    'Demand_Forecast': 'demand_forecast'
})

In [ ]:
# Engineered columns (from EDA findings)

df['reorder_breach'] = (df['inventory_level'] <= df['reorder_point']).astype(int)
df['forecast_error'] = df['demand_forecast'] - df['units_sold']
df['abs_pct_error'] = np.where(
    df['units_sold'] > 0,
    (df['forecast_error'].abs() / df['units_sold']) ,
    np.nan
)
df['margin'] = df['unit_price'] - df['unit_cost']
df['margin_pct'] = np.where(df['unit_price'] > 0, (df['margin'] / df['unit_price']) , np.nan)

In [ ]:
# Data quality assertions

assert df['sku_id'].notnull().all(), "Null SKU_ID found"
assert (df['inventory_level'] >= 0).all(), "Negative inventory found"
assert df.duplicated(subset=['sale_date', 'sku_id', 'warehouse_id']).sum() == 0, "Duplicate grain found"
print(f"✅ {len(df)} rows passed quality checks, ready to load")

✅ 91250 rows passed quality checks, ready to load


In [ ]:
# Load environment variables
load_dotenv()

server = os.getenv('DB_SERVER')
database = os.getenv('DB_NAME')
username = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')

# Create connection string
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    f"Encrypt=yes;"
    f"TrustServerCertificate=no;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)

# Load cleaned DataFrame 

print("Starting data load...")

df.to_sql(
    name='stg_inventory',
    con=engine,
    if_exists='replace',      
    index=False,
    chunksize=1000            
)

print("✅ Data loaded successfully into stg_inventory!")

with engine.connect() as conn:
    result = conn.exec_driver_sql("SELECT COUNT(*) FROM stg_inventory").fetchone()
    print("Row count in Azure:", result[0])

Starting data load...
✅ Data loaded successfully into stg_inventory!
Row count in Azure: 91250
